In [2]:
import requests
from bs4 import BeautifulSoup
import re
import time
from urllib.parse import urlparse, urljoin
from typing import Optional

def is_valid_url(url: str) -> bool:
    """Checks if a URL is syntactically valid."""
    try:
        result = urlparse(url)
        return all([result.scheme, result.netloc])
    except ValueError:
        return False

def find_careers_page(company_name: str) -> Optional[str]:
    """
    Attempts to find the careers page URL for a given company name.

    Args:
        company_name: The name of the company.

    Returns:
        The URL of the careers page if found, otherwise None.
    """
    company_name_lower = company_name.lower().replace(' ', '')
    base_url = f"https://www.{company_name_lower}.com"
    possible_paths = ["/careers", "/jobs", "/about-us/careers", "/join-us", "/careers-jobs"]
    possible_subdomains = ["careers.", "jobs."]

    # Construct potential URLs
    potential_urls = [base_url + path for path in possible_paths]
    potential_urls.extend([base_url.replace("www.", subdomain) for subdomain in possible_subdomains if is_valid_url(base_url.replace("www.", subdomain) + possible_paths[0])])
    potential_urls.append(f"https://{company_name_lower}.com/careers") #adding a general case

    for url in potential_urls:
        if is_valid_url(url):
            try:
                headers = {
                    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
                }  # Add User-Agent
                response = requests.get(url, headers=headers, timeout=5)
                response.raise_for_status()  # Raise an exception for bad status codes
                # Check for keywords in the URL and the page content
                if "careers" in response.url.lower() or "jobs" in response.url.lower() or "careers" in response.text.lower() or "jobs" in response.text.lower():
                    return response.url  # Return the actual URL

            except requests.exceptions.RequestException as e:
                print(f"Error accessing {url}: {e}")
                time.sleep(1)  # Be polite and add a delay

    # Try a more generic search if direct paths fail
    search_url = f"https://www.google.com/search?q=site:{base_url} careers jobs employment"
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }  # Add User-Agent
        search_response = requests.get(search_url, headers=headers, timeout=10)
        search_response.raise_for_status()
        soup = BeautifulSoup(search_response.content, 'html.parser')
        for link in soup.find_all('a', href=True):
            href = link['href']
            if "careers" in href.lower() or "jobs" in href.lower() and base_url in href:
                if not href.startswith("https://www.google.com"):
                    return href.split('&')[0]  # Clean up Google's URL parameters
    except requests.exceptions.RequestException as e:
        print(f"Error during Google search: {e}")
    except Exception as e:
        print(f"An error occurred during search: {e}")

    return None  # Return None if no careers page is found

def sanitize_title(title: str) -> str:
    """
    Sanitizes a job title for comparison by removing special characters and extra spaces.

    Args:
        title: The job title to sanitize.

    Returns:
        The sanitized job title.
    """
    title = re.sub(r'[^a-zA-Z0-9\s]', '', title)  # Remove special chars
    title = re.sub(r'\s+', ' ', title).strip()  # Remove extra spaces
    return title.lower()

def validate_job_posting(job_title: str, company_name: str) -> str:
    """
    Checks if the job title exists on the company's careers page.

    Args:
        job_title: The job posting title from LinkedIn.
        company_name: The name of the company.

    Returns:
        A string indicating the validation result.
    """
    careers_url = find_careers_page(company_name)

    if not careers_url:
        return "Could not automatically find the company's careers page. Please check manually."

    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }  # Add User-Agent
        response = requests.get(careers_url, headers=headers, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')

        # Sanitize job title and page text for comparison
        job_title_sanitized = sanitize_title(job_title)
        page_text_sanitized = sanitize_title(soup.get_text(separator=' '))

        if job_title_sanitized in page_text_sanitized:
            return f"'{job_title}' found on the company's careers page: {careers_url}"
        else:
            return f"'{job_title}' NOT found on the company's careers page: {careers_url}. It might be a fake posting or listed under a different title."

    except requests.exceptions.RequestException as e:
        return f"Error accessing the careers page ({careers_url}): {e}"
    except Exception as e:
        return f"An error occurred during scraping: {e}"

def run_validation():
    """
    Runs the job posting validation process.  Wrapped in a function
    """
    # User Input
    job_title_input = input("Enter the job posting title from LinkedIn: ")
    company_name_input = input("Enter the name of the company: ")

    # Validate and Print Result
    validation_result = validate_job_posting(job_title_input, company_name_input)
    print("\nValidation Result:")
    print(validation_result)

if __name__ == "__main__":
    run_validation() # Added main block
# Explanation and Improvements:Import Libraries: Imports the necessary libraries.is_valid_url(url): Checks if a given URL is syntactically valid.find_careers_page(company_name):Attempts to locate a company's career page.Constructs potential URL patterns.Sends HTTP requests with a User-Agent header to mimic a browser.Handles potential HTTP errors.Includes a broader search using Google as a fallback.Implements a delay to prevent overloading servers.Now includes more paths and subdomains.sanitize_title(title):Sanitizes the job title by:Removing special characters.Collapsing multiple spaces into single spaces.Converting to lowercase for case-insensitive comparison.validate_job_posting(job_title, company_name):Fetches the career page URL.Scrapes the page content.Sanitizes both the input job title and the scraped page text.Checks if the sanitized job title is present in the sanitized page text.Returns the result.run_validation(): Encapsulates the user input and validation process.if __name__ == "__main__":: Ensures that the run_validation() function is called when the script is executed.  This is good practice.User-Agent Header: The code now includes a User-Agent header in the requests. This helps prevent websites from blocking your requests, as they can identify automated scripts without it.Error Handling: The code includes more robust error handling (e.g., checking for valid URLs, handling HTTP errors).URL Validation: The is_valid_url function ensures that the generated URLs are valid before making requests.Sanitized Comparison: The  sanitize_title function ensures that the job title comparison is more robust.Type Hints: The code uses type hints for better readability and maintainability.Comments and Structure: The code is well-commented and organized into functions for better readability and maintainability.How to Use:Install Libraries:pip install requests beautifulsoup4
# Run the Notebook: Execute the code in a Jupyter Notebook cell.Enter Input: Provide the job title from LinkedIn and the company name when prompted.View Results: The program will output whether the job title was found on the company's career page.

Enter the job posting title from LinkedIn:  Associate Product Manager
Enter the name of the company:  Motion Recruitment


Error accessing https://www.motionrecruitment.com/careers: 404 Client Error: Not Found for url: https://motionrecruitment.com/careers
Error accessing https://www.motionrecruitment.com/jobs: 404 Client Error: Not Found for url: https://motionrecruitment.com/jobs
Error accessing https://www.motionrecruitment.com/about-us/careers: 404 Client Error: Not Found for url: https://motionrecruitment.com/about-us/careers
Error accessing https://www.motionrecruitment.com/join-us: 404 Client Error: Not Found for url: https://motionrecruitment.com/join-us
Error accessing https://www.motionrecruitment.com/careers-jobs: 404 Client Error: Not Found for url: https://motionrecruitment.com/careers-jobs
Error accessing https://careers.motionrecruitment.com: HTTPSConnectionPool(host='careers.motionrecruitment.com', port=443): Max retries exceeded with url: / (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x11a9bd4b0>: Failed to establish a new connection: [Errno 8] nodename nor